In [1]:
import pandas as pd
import webbrowser
from fyers_apiv3 import fyersModel
from datetime import datetime, timedelta
import time

In [2]:
client_id="W7LM9VJA41-200"
secret_key="qpKCdzLr6qijQPIR"
redirect_uri="https://tarantularesearch.com/"

In [3]:
def generate_fyers_access_token(
    client_id: str,
    secret_key: str,
    redirect_uri: str,
    state: str = "sample_state"
) -> dict:
    """
    Handles FYERS OAuth flow and returns access token response.

    Returns
    -------
    dict
        Full response containing access_token, refresh_token, expiry, etc.
    """

    # -------------------------------
    # Step 1: Generate auth URL
    # -------------------------------
    session = fyersModel.SessionModel(
        client_id=client_id,
        secret_key=secret_key,
        redirect_uri=redirect_uri,
        response_type="code",
        state=state
    )

    auth_url = session.generate_authcode()
    webbrowser.open(auth_url)

    # -------------------------------
    # Step 2: User pastes redirected URL
    # -------------------------------
    raw_code = input("Paste redirected URL here: ").strip()

    try:
        auth_code = raw_code.split("auth_code=")[1].split("&")[0]
    except IndexError:
        raise ValueError("Invalid redirect URL. Auth code not found.")

    # -------------------------------
    # Step 3: Exchange auth_code for access_token
    # -------------------------------
    session = fyersModel.SessionModel(
        client_id=client_id,
        secret_key=secret_key,
        redirect_uri=redirect_uri,
        response_type="code",
        grant_type="authorization_code"
    )

    session.set_token(auth_code)
    token_response = session.generate_token()

    return token_response


In [4]:


token_response = generate_fyers_access_token(
    client_id=client_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri
)

access_token = token_response["access_token"]
print("Access Token:", access_token)


Access Token: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJhdWQiOlsiZDoxIiwiZDoyIiwieDowIiwieDoxIiwieDoyIl0sImF0X2hhc2giOiJnQUFBQUFCcDJLWTN2SnpuUDVFNHZJdE0zelFBY2lCWmtyZHhhUWdReFBIWjVmcS1iTUh4MjJmYmRndXp3ckNRSGdyS196SmhFSmNTTHBmVkxsMmU5S0t3NWpIdzJGMTJzdGxGa2hIT25vckVhWnVTcDRzbERLbz0iLCJkaXNwbGF5X25hbWUiOiIiLCJvbXMiOiJLMSIsImhzbV9rZXkiOiIwZjk0MWIyMzk2NTZkZjdjNGQ3MzZmOTk4NTE2M2ZjZjQxODRiOWJiOTk1MzAxNWI4ZWNiOTdlMCIsImlzRGRwaUVuYWJsZWQiOiJOIiwiaXNNdGZFbmFibGVkIjoiTiIsImZ5X2lkIjoiWEwwMDcyMSIsImFwcFR5cGUiOjIwMCwiZXhwIjoxNzc1ODY3NDAwLCJpYXQiOjE3NzU4MDYwMDcsImlzcyI6ImFwaS5meWVycy5pbiIsIm5iZiI6MTc3NTgwNjAwNywic3ViIjoiYWNjZXNzX3Rva2VuIn0.IZjBSSC2b-rVOvaYND8AbE79AK0mfYMOJfHOmSmugpM


In [5]:
client_id = client_id
# Initialize the FyersModel instance with your client_id, access_token, and enable async mode
fyers = fyersModel.FyersModel(client_id=client_id, token=access_token,is_async=False, log_path="")

data = {
    "symbol":"NSE:IDEA-EQ",
    "qty":1,
    "type":1,
    "side":1,
    "productType":"INTRADAY",
    "limitPrice":0,
    "stopPrice":0,
    "validity":"DAY",
    "disclosedQty":0,
    "offlineOrder":False,
    "orderTag":"tag1",
    "isSliceOrder":False
}
response = fyers.place_order(data=data)
print(response)


{'code': -50, 'message': 'Request rejected: Orders are only allowed from whitelisted IP addresses. This request was received from IP: 2401:4900:1c61:5cdd:8489:747d:ec87:31a2.', 's': 'error'}


In [6]:
access_token

'eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJhdWQiOlsiZDoxIiwiZDoyIiwieDowIiwieDoxIiwieDoyIl0sImF0X2hhc2giOiJnQUFBQUFCcDJLWTN2SnpuUDVFNHZJdE0zelFBY2lCWmtyZHhhUWdReFBIWjVmcS1iTUh4MjJmYmRndXp3ckNRSGdyS196SmhFSmNTTHBmVkxsMmU5S0t3NWpIdzJGMTJzdGxGa2hIT25vckVhWnVTcDRzbERLbz0iLCJkaXNwbGF5X25hbWUiOiIiLCJvbXMiOiJLMSIsImhzbV9rZXkiOiIwZjk0MWIyMzk2NTZkZjdjNGQ3MzZmOTk4NTE2M2ZjZjQxODRiOWJiOTk1MzAxNWI4ZWNiOTdlMCIsImlzRGRwaUVuYWJsZWQiOiJOIiwiaXNNdGZFbmFibGVkIjoiTiIsImZ5X2lkIjoiWEwwMDcyMSIsImFwcFR5cGUiOjIwMCwiZXhwIjoxNzc1ODY3NDAwLCJpYXQiOjE3NzU4MDYwMDcsImlzcyI6ImFwaS5meWVycy5pbiIsIm5iZiI6MTc3NTgwNjAwNywic3ViIjoiYWNjZXNzX3Rva2VuIn0.IZjBSSC2b-rVOvaYND8AbE79AK0mfYMOJfHOmSmugpM'

    Tradebook
- Fetches all trades for the current day

In [7]:
from fyers_apiv3 import fyersModel

client_id = client_id
access_token = access_token
# Initialize the FyersModel instance with your client_id, access_token, and enable async mode
fyers = fyersModel.FyersModel(client_id=client_id, token=access_token,is_async=False, log_path="")

response = fyers.tradebook()
print(response)

{'code': 200, 'message': '', 's': 'ok', 'tradeBook': []}


In [8]:
response["tradeBook"]

[]

In [9]:
# Initialize the FyersModel instance with your client_id, access_token, and enable async mode
fyers = fyersModel.FyersModel(client_id=client_id, token=access_token,is_async=False, log_path="")

response = fyers.orderbook()
print(response)

{'code': 200, 'message': '', 's': 'ok', 'orderBook': []}


    Orders

- For current day

In [10]:
response["orderBook"]

[]

    Positions
-   Fetches current day's open and closed positions

In [11]:
# Initialize the FyersModel instance with your client_id, access_token, and enable async mode
fyers = fyersModel.FyersModel(client_id=client_id, token=access_token,is_async=False, log_path="")

response = fyers.positions()
print(response)

{'code': 200, 'message': '', 's': 'ok', 'netPositions': [], 'overall': {'count_open': 0, 'count_total': 0, 'pl_realized': 0, 'pl_total': 0, 'pl_unrealized': 0}}


In [12]:
response

{'code': 200,
 'message': '',
 's': 'ok',
 'netPositions': [],
 'overall': {'count_open': 0,
  'count_total': 0,
  'pl_realized': 0,
  'pl_total': 0,
  'pl_unrealized': 0}}

In [13]:
response["overall"]["count_open"]

0

In [14]:
response["netPositions"]

[]

In [15]:
data = {
    "symbols":"NSE:TMPV-EQ"
}
response = fyers.quotes(data=data)

{'code': 200,
 'message': '',
 's': 'ok',
 'netPositions': [{'symbol': 'NSE:TMPV-EQ',
   'id': 'NSE:TMPV-EQ-INTRADAY',
   'buyAvg': 305.175,
   'buyQty': 10,
   'buyVal': 3051.75,
   'sellAvg': 304.5,
   'sellQty': 5,
   'sellVal': 1522.5,
   'netAvg': 305.175,
   'netQty': 5,
   'side': 1,
   'qty': 5,
   'productType': 'INTRADAY',
   'realized_profit': -3.375000000000057,
   'crossCurrency': '',
   'rbiRefRate': 1,
   'fyToken': '10100000003456',
   'exchange': 10,
   'segment': 10,
   'dayBuyQty': 10,
   'daySellQty': 5,
   'cfBuyQty': 0,
   'cfSellQty': 0,
   'qtyMulti_com': 1,
   'pl': -0.7500000000001705,
   'unrealized_profit': 2.6249999999998863,
   'ltp': 305.7,
   'slNo': 0}],
 'overall': {'count_open': 1,
  'count_total': 1,
  'pl_realized': -3.375000000000057,
  'pl_total': -0.7500000000001705,
  'pl_unrealized': 2.6249999999998863}}

[{'symbol': 'NSE:TMPV-EQ',
  'id': 'NSE:TMPV-EQ-INTRADAY',
  'buyAvg': 304.6,
  'buyQty': 5,
  'buyVal': 1523,
  'sellAvg': 304.5,
  'sellQty': 5,
  'sellVal': 1522.5,
  'netAvg': 0,
  'netQty': 0,
  'side': 0,
  'qty': 0,
  'productType': 'INTRADAY',
  'realized_profit': -0.5000000000001137,
  'crossCurrency': '',
  'rbiRefRate': 1,
  'fyToken': '10100000003456',
  'exchange': 10,
  'segment': 10,
  'dayBuyQty': 5,
  'daySellQty': 5,
  'cfBuyQty': 0,
  'cfSellQty': 0,
  'qtyMulti_com': 1,
  'pl': -0.5000000000001137,
  'unrealized_profit': 0,
  'ltp': 305.7,
  'slNo': 0}]

{'count_open': 0,
 'count_total': 1,
 'pl_realized': -1.500000000000341,
 'pl_total': -1.500000000000341,
 'pl_unrealized': 0}

[{'symbol': 'NSE:TMPV-EQ',
  'id': 'NSE:TMPV-EQ-INTRADAY',
  'buyAvg': 305.175,
  'buyQty': 10,
  'buyVal': 3051.75,
  'sellAvg': 305.025,
  'sellQty': 10,
  'sellVal': 3050.25,
  'netAvg': 0,
  'netQty': 0,
  'side': 0,
  'qty': 0,
  'productType': 'INTRADAY',
  'realized_profit': -1.500000000000341,
  'crossCurrency': '',
  'rbiRefRate': 1,
  'fyToken': '10100000003456',
  'exchange': 10,
  'segment': 10,
  'dayBuyQty': 10,
  'daySellQty': 10,
  'cfBuyQty': 0,
  'cfSellQty': 0,
  'qtyMulti_com': 1,
  'pl': -1.500000000000341,
  'unrealized_profit': 0,
  'ltp': 305.55,
  'slNo': 0}]

In [16]:
response

{'message': '',
 'code': 200,
 'd': [{'n': 'NSE:TMPV-EQ',
   'v': {'ask': 339.25,
    'bid': 339.2,
    'chp': 1.79,
    'ch': 5.95,
    'description': 'NSE:TMPV-EQ',
    'exchange': 'NSE',
    'fyToken': '10100000003456',
    'high_price': 339.8,
    'low_price': 335,
    'lp': 339.2,
    'open_price': 336.6,
    'original_name': 'NSE:TMPV-EQ',
    'prev_close_price': 333.25,
    'short_name': 'TMPV-EQ',
    'spread': 0.05,
    'symbol': 'NSE:TMPV-EQ',
    'tt': '1775779200',
    'volume': 5903134,
    'atp': 338.02},
   's': 'ok'}],
 's': 'ok'}